# Bell-Plate FEM — interactive modal analysis

Free-vibration (modal) analysis of a rectangular **bell plate** (a struck metal idiophone)
with optional **triangular edge incuts**, using 3D solid finite elements
(gmsh mesh → scikit-fem P2 tetrahedra → free-free eigenvalue solve → matplotlib mode shapes).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy.sparse.linalg import eigsh

import os, tempfile, traceback
import gmsh
import skfem
from skfem import MeshTet, Basis, ElementVector, ElementTetP2, BilinearForm
from skfem.helpers import ddot, sym_grad, trace, dot
import ipywidgets as W
from IPython.display import display

print("scikit-fem", skfem.__version__, "| gmsh", gmsh.__version__, "| ipywidgets", W.__version__)

## Model details

- **Boundary condition:** *free-free* (unconstrained, as if suspended at nodal points and
  struck). A free body has **6 rigid-body modes** at ~0 Hz, detected and filtered out.
- **Units:** everything is SI internally (m, Pa, kg/m³); you enter mm / GPa, converted on the way in.
- **Mesh sizing** auto-derives the element size from `f_max` (flexural wavelength ÷ density),
  capped to **one quadratic (P2) element through the thickness** (within ~1% of a finer mesh but
  several times faster) and a couple of elements across each notch.
- **Incuts:** isosceles V-notches cut through the full thickness, boolean-subtracted in gmsh.

In [ ]:
# Material presets: E [Pa], rho [kg/m^3], nu [-]
MATERIALS = {
    "steel":    dict(E=2.10e11, rho=7850.0, nu=0.30),
    "bronze":   dict(E=1.05e11, rho=8800.0, nu=0.34),
    "aluminum": dict(E=6.90e10, rho=2700.0, nu=0.33),
}

def flexural_wavelength(f, E, rho, nu, H):
    '''Bending wavelength of a plate of thickness H at frequency f (thin-plate estimate).'''
    D = E*H**3 / (12.0*(1.0 - nu**2))
    omega = 2.0*np.pi*f
    kappa = (omega**2 * rho*H / D)**0.25       # D kappa^4 = rho H omega^2
    return 2.0*np.pi/kappa

def compute_element_size(L, W_, H, E, rho, nu, f_max, epw, notches_m, holes_m=None):
    '''Choose the (bulk) mesh element size; return (h_elem, governing_rule, printable_lines).

    Sizing rules (smallest wins):
      - in-plane: flexural wavelength at f_max / epw (resolves the bending waves);
      - thickness: <= H, i.e. one QUADRATIC (P2) element through the thickness. A P2 element
        has a mid-edge node, so a single element already captures the through-thickness bending
        of the low modes -- verified within ~1% of a 2-elements-through-thickness mesh while
        solving several times faster. (Use the convergence check for a finer confirmation.)
      - notches: <= min(base, depth) / 2, so each notch spans >= ~2 elements.
    Holes do NOT shrink this bulk size: their circular boundaries are refined LOCALLY in
    make_mesh (curvature-based), so a small hole no longer forces the whole plate fine
    (that was a ~5x DOF / solve-time blow-up).'''
    lam     = flexural_wavelength(f_max, E, rho, nu, H)
    h_wave  = lam / epw
    h_thick = H                                  # one P2 element through the thickness
    h = min(h_wave, h_thick)
    rule = "thickness cap (H)" if h_thick <= h_wave else f"{epw}/flexural wavelength"
    lines = [f"Flexural wavelength @ {f_max:g} Hz : {lam*1e3:7.1f} mm",
             f"  wavelength rule  -> h_wave    : {h_wave*1e3:7.2f} mm",
             f"  thickness rule   -> H         : {h_thick*1e3:7.2f} mm"]
    if notches_m:
        feat = min(min(n['base'], n['depth']) for n in notches_m)
        h_feat = feat / 2.0
        lines.append(f"  notch rule       -> feat/2    : {h_feat*1e3:7.2f} mm")
        if h_feat < h:
            h, rule = h_feat, "notch feature (min(base,depth)/2)"
    if holes_m:
        lines.append(f"  holes            -> refined locally near each hole (bulk size unchanged)")
    n_est = (L/h)*(W_/h)*(H/h)*6.0
    lines.append(f"  chosen element size  h_elem   : {h*1e3:7.2f} mm   (governed by {rule})")
    lines.append(f"  rough element count           : ~{n_est:,.0f} tets (plus local refinement at holes)")
    return h, rule, lines

In [ ]:
def notch_prism(pos, base, depth, edge, L, W_, H):
    '''Build one triangular-prism cutting tool in the active gmsh model; return its (3, tag).'''
    if   edge == "bottom": O = np.array([pos, 0.0]); n = np.array([0.0,  1.0]); t = np.array([1.0, 0.0])
    elif edge == "top":    O = np.array([pos, W_]);  n = np.array([0.0, -1.0]); t = np.array([1.0, 0.0])
    elif edge == "left":   O = np.array([0.0, pos]); n = np.array([ 1.0, 0.0]); t = np.array([0.0, 1.0])
    elif edge == "right":  O = np.array([L, pos]);   n = np.array([-1.0, 0.0]); t = np.array([0.0, 1.0])
    else: raise ValueError(f"unknown edge {edge!r}")

    over = 0.05*depth                          # overhang so the base pokes outside the plate
    A  = O + depth*n                           # apex (inside the plate)
    P1 = O - 0.5*base*t                        # base endpoints at the surface
    P2 = O + 0.5*base*t
    s  = (depth + over)/depth                  # scale base outward from the apex
    P1o, P2o = A + s*(P1 - A), A + s*(P2 - A)

    pt = [gmsh.model.occ.addPoint(p[0], p[1], -0.05*H) for p in (A, P1o, P2o)]
    ln = [gmsh.model.occ.addLine(pt[i], pt[(i+1) % 3]) for i in range(3)]
    surf = gmsh.model.occ.addPlaneSurface([gmsh.model.occ.addCurveLoop(ln)])
    out = gmsh.model.occ.extrude([(2, surf)], 0.0, 0.0, H + 0.10*H)   # through the top face
    return [e for e in out if e[0] == 3][0]

def hole_cylinder(x, y, diameter, H):
    '''Build one through-thickness cylindrical cutting tool; return its (3, tag).'''
    tag = gmsh.model.occ.addCylinder(x, y, -0.05*H, 0.0, 0.0, H + 0.10*H, diameter/2.0)  # z-overhang
    return (3, tag)

def make_mesh(L, W_, H, h, notches=None, holes=None):
    '''Mesh an L x W x H box, optionally with triangular edge notches and/or cylindrical holes
    cut out; return MeshTet.

    `h` is the BULK element size. Holes are resolved by LOCAL curvature-based refinement (only
    near each circular boundary), so the bulk stays coarse — a small hole no longer forces the
    whole plate fine (~5x fewer DOFs / faster solve, frequencies unchanged to <0.2%).

    Raises a clear RuntimeError if the geometry leaves no valid solid, or if meshing produces no
    tetrahedra, instead of failing obscurely downstream.'''
    gmsh.initialize()
    try:
        gmsh.option.setNumber("General.Terminal", 0)
        gmsh.model.add("plate")
        box = gmsh.model.occ.addBox(0.0, 0.0, 0.0, L, W_, H)
        tools = []
        if notches:
            tools += [notch_prism(nc["pos"], nc["base"], nc["depth"], nc["edge"], L, W_, H)
                      for nc in notches]
        if holes:
            tools += [hole_cylinder(hl["x"], hl["y"], hl["diameter"], H) for hl in holes]
        if tools:
            gmsh.model.occ.cut([(3, box)], tools, removeTool=True)
        gmsh.model.occ.synchronize()
        if len(gmsh.model.getEntities(3)) == 0:
            raise RuntimeError("The cuts removed the entire plate (no solid left) — "
                               "reduce notch/hole sizes, or move overlapping cuts apart.")
        gmsh.option.setNumber("Mesh.MeshSizeMax", h)
        if holes:
            # Refine ONLY around the curved hole boundaries (the only curved entities here),
            # grading back up to the coarse bulk size away from them.
            d_min = min(hl["diameter"] for hl in holes)
            gmsh.option.setNumber("Mesh.MeshSizeMin", min(h, max(3.0e-4, d_min/8.0)))
            gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 14)   # ~14 elements around each hole
        else:
            gmsh.option.setNumber("Mesh.MeshSizeMin", h)
        gmsh.model.mesh.generate(3)
        path = os.path.join(tempfile.gettempdir(), "bell_plate.msh")
        gmsh.write(path)
    finally:
        gmsh.finalize()
    mesh = MeshTet.load(path)
    if mesh.t.shape[1] == 0:
        raise RuntimeError("Meshing produced no tetrahedra — check the plate/notch/hole dimensions.")
    return mesh

In [ ]:
def solve_modes(mesh, E, rho, nu, n_modes, f_max):
    '''Assemble 3D elasticity (P2) and solve the free-free modal problem.

    Returns (basis, freqs, vecs) sorted ascending, INCLUDING the 6 rigid-body modes.'''
    mu  = E / (2.0*(1.0 + nu))
    lam = E*nu / ((1.0 + nu)*(1.0 - 2.0*nu))
    basis = Basis(mesh, ElementVector(ElementTetP2()))

    @BilinearForm
    def stiffness(u, v, w):
        return 2.0*mu*ddot(sym_grad(u), sym_grad(v)) + lam*trace(sym_grad(u))*trace(sym_grad(v))

    @BilinearForm
    def mass(u, v, w):
        return rho*dot(u, v)

    K = stiffness.assemble(basis)
    M = mass.assemble(basis)

    sigma = -(2.0*np.pi*max(1.0, 0.05*f_max))**2   # negative -> (K - sigma M) is SPD
    k = min(n_modes + 6 + 4, K.shape[0] - 2)
    if k < 1:
        raise RuntimeError("Mesh is too small to solve for any modes — use a finer mesh.")

    # eigsh (ARPACK shift-invert) occasionally fails to converge; retry once with a larger
    # Krylov subspace and more iterations before giving up with a clear message.
    try:
        vals, vecs = eigsh(K, k=k, M=M, sigma=sigma, which="LM")
    except Exception:
        ncv = min(K.shape[0] - 1, max(2*k + 1, 40))
        try:
            vals, vecs = eigsh(K, k=k, M=M, sigma=sigma, which="LM", ncv=ncv, maxiter=5000)
        except Exception as err:
            raise RuntimeError(
                f"Eigensolver did not converge (k={k}, ndof={K.shape[0]}). "
                f"Try fewer modes, or a coarser/finer mesh. [{type(err).__name__}: {err}]")

    order = np.argsort(vals)
    vals, vecs = vals[order], vecs[:, order]
    freqs = np.sqrt(np.clip(vals, 0.0, None)) / (2.0*np.pi)
    return basis, freqs, vecs

def split_rigid_elastic(freqs, n_modes):
    '''Separate the 6 rigid-body modes (f ~ 0) from elastic modes by a relative threshold.

    Robust to receiving fewer than 7 modes; returns (n_rigid, elastic_index_array).'''
    freqs = np.asarray(freqs, dtype=float)
    if freqs.size == 0:
        return 0, np.array([], dtype=int)
    ref = freqs[6] if freqs.size > 6 else freqs[-1]          # first clearly-elastic frequency
    thr = max(1e-3, 0.01 * ref)                              # rigid modes sit far below this
    rigid_mask = freqs < thr
    elastic_idx = np.where(~rigid_mask)[0][:n_modes]
    return int(rigid_mask.sum()), elastic_idx

def elastic_frequencies(mesh, E, rho, nu, n_modes, f_max):
    '''Convenience: solve and return just the elastic frequency array.'''
    _, freqs, _ = solve_modes(mesh, E, rho, nu, n_modes, f_max)
    _, idx = split_rigid_elastic(freqs, n_modes)
    return freqs[idx]

In [ ]:
def _top_surface_triangulation(mesh, H):
    '''Triangulation of the top (z=H) surface from real mesh facets (respects notch cut-outs).'''
    f = mesh.facets
    top = np.all(np.abs(mesh.p[2][f] - H) < 1e-6*H, axis=0)
    facets = f[:, top]
    nodes = np.unique(facets)
    remap = -np.ones(mesh.p.shape[1], dtype=int)
    remap[nodes] = np.arange(nodes.size)
    tri = mtri.Triangulation(mesh.p[0][nodes], mesh.p[1][nodes], remap[facets].T)
    return nodes, tri

def _draw_mode_contour(ax, mesh, uz, H):
    '''Filled contour of normalised u_z on the top surface, drawn into ax.

    Colour only (no colorbar): blue = down, red = up, white = nodal line (u_z = 0). The colour
    scale is symmetric (-1..1) so white sits exactly at zero. Axis ticks are dropped to keep the
    grid clean.'''
    uzn = uz / (np.max(np.abs(uz)) or 1.0)
    nodes, tri = _top_surface_triangulation(mesh, H)
    levels = np.linspace(-1.0, 1.0, 21)
    ax.tricontourf(tri, uzn[nodes], levels=levels, cmap="RdBu_r",
                   norm=plt.Normalize(-1.0, 1.0), extend="both")
    ax.tricontour(tri, uzn[nodes], levels=[0.0], colors="k", linewidths=0.7)
    ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])

def plot_modes_grid(basis, mesh, vecs, freqs, L, W_, H, f_max=None, ncols=3):
    '''Show every elastic mode shape as a 2D top-surface u_z contour, laid out in a grid.

    Defaults to 3 columns, so the usual 6 modes land in a 2x3 grid; it adapts to any count
    (fewer columns if there are fewer modes, more rows if there are more). No colorbar — the
    colours alone carry the sign (blue = down, red = up, white = nodal line). A mode above f_max
    gets a ⚠ in its title.

    Built with plt.Figure (unmanaged) so display() renders the whole grid exactly once.'''
    n = len(freqs)
    if n == 0:
        print("(no elastic modes to plot)")
        return
    ncols = max(1, min(ncols, n))
    nrows = int(np.ceil(n / ncols))
    fig = plt.Figure(figsize=(3.5*ncols, 2.8*nrows), constrained_layout=True)
    for i in range(n):
        ax = fig.add_subplot(nrows, ncols, i + 1)
        uz = vecs[:, i][basis.nodal_dofs[2]]
        _draw_mode_contour(ax, mesh, uz, H)
        flag = "  ⚠" if (f_max is not None and freqs[i] > f_max) else ""
        ax.set_title(f"Mode {i+1}:  {freqs[i]:.1f} Hz{flag}", fontsize=10)
    display(fig)

In [ ]:
def plot_mesh_3d(mesh, L, W_, H):
    '''Quick 3D look at the surface mesh (boundary facets only — interior tets are skipped) so
    you can eyeball the element density and where it's refined (e.g. around holes/notches).

    Uses plt.Figure (unmanaged) so display() renders it exactly once — no inline auto-display
    duplicate.'''
    fac = mesh.facets[:, mesh.boundary_facets()]          # outer skin triangles
    verts = np.transpose(mesh.p[:, fac], (2, 1, 0))
    fig = plt.Figure(figsize=(7.5, 4.5), constrained_layout=True)
    ax = fig.add_subplot(111, projection="3d")
    ax.add_collection3d(Poly3DCollection(verts, facecolor="#cfe0f0", edgecolor="#37547a",
                                         linewidths=0.3))
    ax.set_xlim(0, L); ax.set_ylim(0, W_); ax.set_zlim(0, H)
    ax.set_box_aspect((L, W_, H))                          # true 1:1:1 scaling (equal aspect)
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
    ax.set_title(f"Boundary mesh — {mesh.p.shape[1]} nodes, {mesh.t.shape[1]} tetrahedra")
    display(fig)


In [ ]:
def _top_boundary_segments(tri):
    '''Boundary edges of a triangulation (edges used by only one triangle) as an (N, 2, 2) array
    of line segments. For the top-surface mesh this traces the real outline — outer edge with
    notch cut-outs, plus the rim of every hole.'''
    e = np.sort(np.concatenate([tri.triangles[:, [0, 1]],
                                tri.triangles[:, [1, 2]],
                                tri.triangles[:, [2, 0]]], axis=0), axis=1)
    uniq, counts = np.unique(e, axis=0, return_counts=True)
    b = uniq[counts == 1]                                   # edges belonging to a single triangle
    return np.stack([np.c_[tri.x[b[:, 0]], tri.y[b[:, 0]]],
                     np.c_[tri.x[b[:, 1]], tri.y[b[:, 1]]]], axis=1)

def plot_nodal_lines(basis, mesh, vecs, freqs, L, W_, H):
    '''Top view of the plate with the nodal line (u_z = 0) of EVERY elastic mode overlaid, each
    mode in its own thick colour, with a legend. The real top outline (outer edge with notch
    cut-outs, plus every hole rim) is drawn in grey underneath. Lets you compare at a glance where
    the different modes have their stationary lines.

    Uses plt.Figure (unmanaged) so display() renders it exactly once.'''
    from matplotlib.lines import Line2D
    from matplotlib.collections import LineCollection
    n = len(freqs)
    if n == 0:
        print("(no modes to draw nodal lines for)")
        return
    nodes, tri = _top_surface_triangulation(mesh, H)
    cmap = plt.cm.tab10 if n <= 10 else plt.cm.tab20

    fig = plt.Figure(figsize=(9.5, 7.0*max(W_/L, 0.18) + 1.0))
    ax = fig.add_subplot(111)
    ax.add_collection(LineCollection(_top_boundary_segments(tri), colors="0.4", linewidths=1.5))
    handles = []
    for i in range(n):
        uz = vecs[:, i][basis.nodal_dofs[2]]
        uzn = uz / (np.max(np.abs(uz)) or 1.0)
        color = cmap(i % cmap.N)
        ax.tricontour(tri, uzn[nodes], levels=[0.0], colors=[color], linewidths=2.8)
        handles.append(Line2D([0], [0], color=color, lw=2.8,
                              label=f"Mode {i+1}: {freqs[i]:.0f} Hz"))
    ax.set_aspect("equal")
    ax.autoscale_view()
    ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
    ax.set_title("Nodal lines ($u_z = 0$) of all modes — top view")
    ax.legend(handles=handles, loc="center left", bbox_to_anchor=(1.02, 0.5),
              fontsize=9, frameon=False)
    fig.subplots_adjust(left=0.09, right=0.74, top=0.92, bottom=0.12)
    display(fig)


In [ ]:
def parse_notches(text):
    '''Parse the notch textarea: one "edge, pos, base, depth" (mm) per line.'''
    out = []
    for raw in text.splitlines():
        s = raw.strip()
        if not s or s.startswith("#"):
            continue
        parts = [p.strip() for p in s.split(",")]
        if len(parts) != 4:
            raise ValueError(f"bad notch line {raw!r} (need: edge, pos, base, depth)")
        edge = parts[0].lower()
        if edge not in ("bottom", "top", "left", "right"):
            raise ValueError(f"bad edge {edge!r} (use bottom/top/left/right)")
        try:
            pos, base, depth = (float(parts[1]), float(parts[2]), float(parts[3]))
        except ValueError:
            raise ValueError(f"notch line {raw!r}: pos/base/depth must be numbers (mm)")
        out.append(dict(edge=edge, pos=pos, base=base, depth=depth))
    return out

def notch_problems(notches_mm, L_mm, W_mm):
    '''List (index, message) for every notch that won't mesh (off-edge / too deep / overlapping /
    non-positive); returns [] if all are valid. Non-raising — used by the live preview.'''
    probs, by_edge = [], {}
    for k, nc in enumerate(notches_mm, 1):
        edge, pos, base, depth = nc["edge"], nc["pos"], nc["base"], nc["depth"]
        if base <= 0 or depth <= 0:
            probs.append((k, f"notch {k} ({edge}): base and depth must be > 0."))
            continue
        along = L_mm if edge in ("bottom", "top") else W_mm   # pos runs along this edge
        into  = W_mm if edge in ("bottom", "top") else L_mm   # depth cuts into this span
        if pos - base/2 < 0 or pos + base/2 > along:
            probs.append((k, f"notch {k} ({edge}): pos={pos:g}, base={base:g} mm runs off the "
                             f"edge (must fit within 0..{along:g} mm)."))
        if depth >= 0.5*into:
            probs.append((k, f"notch {k} ({edge}): depth={depth:g} mm >= half the {into:g} mm span."))
        by_edge.setdefault(edge, []).append((pos, base, k))
    for edge, items in by_edge.items():
        items.sort()
        for (p1, b1, k1), (p2, b2, k2) in zip(items, items[1:]):
            if p2 - p1 < 0.5*(b1 + b2):
                probs.append((k2, f"notches {k1} and {k2} on {edge} overlap "
                                  f"(centres {p1:g} and {p2:g} mm)."))
    return probs

def hole_problems(holes_mm, L_mm, W_mm):
    '''List (index, message) for holes that won't mesh (off-plate / overlapping). Holes with
    diameter <= 0 are treated as "off" and skipped. Non-raising — used by the live preview.'''
    probs = []
    active = [(k, h) for k, h in enumerate(holes_mm, 1) if h["diameter"] > 0]
    for k, h in active:
        x, y, r = h["x"], h["y"], h["diameter"]/2.0
        if x - r < 0 or x + r > L_mm or y - r < 0 or y + r > W_mm:
            probs.append((k, f"hole {k}: ø{2*r:g} mm at ({x:g}, {y:g}) sticks outside the "
                             f"{L_mm:g}x{W_mm:g} mm plate."))
    for i in range(len(active)):
        for j in range(i+1, len(active)):
            k1, h1 = active[i]; k2, h2 = active[j]
            d = ((h1["x"]-h2["x"])**2 + (h1["y"]-h2["y"])**2) ** 0.5
            need = 0.5*(h1["diameter"] + h2["diameter"])
            if d < need:
                probs.append((k2, f"holes {k1} and {k2} overlap (centres {d:.1f} mm apart, "
                                  f"need >= {need:.1f})."))
    return probs

def geometry_problems(notches_mm, holes_mm, L_mm, W_mm):
    '''All notch + hole problems for the current plate (used by preview and validation).'''
    return notch_problems(notches_mm, L_mm, W_mm) + hole_problems(holes_mm, L_mm, W_mm)

def validate_geometry(notches_mm, holes_mm, L_mm, W_mm):
    '''Raise ValueError (listing every problem) if any notch/hole is invalid — before meshing.'''
    probs = geometry_problems(notches_mm, holes_mm, L_mm, W_mm)
    if probs:
        raise ValueError("; ".join(m for _, m in probs))

def run_analysis(L_mm, W_mm, H_mm, material, customE, customRho, customNu,
                 f_max, n_modes, epw, enable_incuts, notches_mm,
                 enable_holes=False, holes_mm=None, do_compare=True):
    '''Full pipeline: size mesh -> mesh -> solve -> report table, plots, comparison.

    Mode shapes are shown as 2D top-surface u_z contours in a grid (no colorbar), followed by a
    single top view overlaying every mode's nodal line.'''
    n_modes = max(1, int(n_modes))
    holes_mm = holes_mm or []
    mat = dict(E=customE, rho=customRho, nu=customNu) if material == "custom" else MATERIALS[material]
    E, rho, nu = mat["E"], mat["rho"], mat["nu"]
    L, W_, H = L_mm/1000.0, W_mm/1000.0, H_mm/1000.0

    active_holes = [h for h in holes_mm if h["diameter"] > 0] if enable_holes else []
    validate_geometry(notches_mm if enable_incuts else [], active_holes, L_mm, W_mm)

    notches_m = ([dict(edge=n["edge"], pos=n["pos"]/1000.0, base=n["base"]/1000.0,
                       depth=n["depth"]/1000.0) for n in notches_mm] if enable_incuts else None)
    holes_m = ([dict(x=h["x"]/1000.0, y=h["y"]/1000.0, diameter=h["diameter"]/1000.0)
                for h in active_holes] if active_holes else None)

    print(f"Plate : {L_mm:g} x {W_mm:g} x {H_mm:g} mm   |   material: {material}  "
          f"(E={E:.3e} Pa, rho={rho:g}, nu={nu:g})")
    print(f"Incuts: {'ON, ' + str(len(notches_m)) + ' notches' if notches_m else 'OFF'}   |   "
          f"Holes: {'ON, ' + str(len(holes_m)) + ' hole(s)' if holes_m else 'OFF'}")

    h_elem, _, _ = compute_element_size(L, W_, H, E, rho, nu, f_max, epw, notches_m, holes_m)

    mesh = make_mesh(L, W_, H, h_elem, notches_m, holes_m)
    print(f"Mesh  : {mesh.p.shape[1]} nodes, {mesh.t.shape[1]} tetrahedra")
    plot_mesh_3d(mesh, L, W_, H)

    basis, freqs, vecs = solve_modes(mesh, E, rho, nu, n_modes, f_max)
    n_rigid, idx = split_rigid_elastic(freqs, n_modes)
    ef, ev = freqs[idx], vecs[:, idx]
    if n_rigid != 6:
        print(f"WARNING: expected 6 rigid-body modes, found {n_rigid} — check inputs.")

    print(f"\nFiltered {n_rigid} rigid-body modes. First {len(ef)} elastic modes:\n")
    print(f"{'mode':>4} | {'freq [Hz]':>11}")
    print("-"*20)
    for i, f in enumerate(ef, 1):
        print(f"{i:>4} | {f:>11.2f}{'  ⚠' if f > f_max else ''}")
    if np.any(ef > f_max):
        print(f"\n⚠ = above f_max ({f_max:g} Hz): under-resolved by this mesh — raise f_max "
              f"(or the mesh density) before trusting these.")

    if (notches_m or holes_m) and do_compare:
        print("\n--- comparison: with vs without cuts ---")
        pf = elastic_frequencies(make_mesh(L, W_, H, h_elem, None, None), E, rho, nu, n_modes, f_max)
        m = min(len(pf), len(ef))
        print(f"{'mode':>4} | {'without cuts [Hz]':>17} | {'with cuts [Hz]':>17} | {'change':>9}")
        print("-"*56)
        for i in range(m):
            print(f"{i+1:>4} | {pf[i]:>17.2f} | {ef[i]:>17.2f} | {100*(ef[i]-pf[i])/pf[i]:>+8.2f}%")

    print("\nMode shapes (top-surface $u_z$ — blue = down, red = up, white = nodal line):")
    plot_modes_grid(basis, mesh, ev, ef, L, W_, H, f_max=f_max, ncols=3)

    print("\nAll nodal lines overlaid (top view):")
    plot_nodal_lines(basis, mesh, ev, ef, L, W_, H)

    print("\nDone.")

In [ ]:
# Fast, mesh-free preview of the actual plate-with-notches-and-holes geometry (live rendering).
from matplotlib.patches import Polygon as _Poly2, Rectangle as _Rect2, Circle as _Circ2

def _notch_triangle(edge, pos, base, depth, L, W_):
    '''2D footprint of one V-notch, matching notch_prism's edge orientation (all in mm).'''
    if   edge == "bottom": O=np.array([pos,0.0]); n=np.array([0.0, 1.0]); t=np.array([1.0,0.0])
    elif edge == "top":    O=np.array([pos,W_]);  n=np.array([0.0,-1.0]); t=np.array([1.0,0.0])
    elif edge == "left":   O=np.array([0.0,pos]); n=np.array([1.0, 0.0]); t=np.array([0.0,1.0])
    elif edge == "right":  O=np.array([L,pos]);   n=np.array([-1.0,0.0]); t=np.array([0.0,1.0])
    else: raise ValueError(f"unknown edge {edge!r}")
    return np.array([O - 0.5*base*t, O + 0.5*base*t, O + depth*n])

def draw_geometry_preview(L_mm, W_mm, H_mm, notches_mm, holes_mm=None):
    '''Draw the plate outline with the notches and holes from the numeric inputs.

    Tolerant: it always draws. Valid cuts are red; any out-of-bounds / overlapping cut is drawn
    orange-dashed (sticking out if it does) and listed underneath, so shrinking the plate just
    flags the offender instead of blanking the picture. (The solve still refuses invalid
    geometry via validate_geometry.)

    Built with plt.Figure (unmanaged) so display() shows it exactly once — no inline
    auto-display duplicate when this is redrawn on every edit inside the Output widget.'''
    L, W_ = float(L_mm), float(W_mm)
    holes_mm = holes_mm or []
    nprobs = notch_problems(notches_mm, L_mm, W_mm)
    hprobs = hole_problems(holes_mm, L_mm, W_mm)
    bad_n = {k for k, _ in nprobs}
    bad_h = {k for k, _ in hprobs}

    fig = plt.Figure(figsize=(7.6, 7.6*max(W_/L, 0.16) + 0.7))
    ax = fig.add_subplot(111)
    ax.add_patch(_Rect2((0, 0), L, W_, fc="#dfe6ee", ec="black", lw=1.6))
    xs, ys = [0.0, L], [0.0, W_]
    for i, nc in enumerate(notches_mm, 1):
        tri = _notch_triangle(nc["edge"], nc["pos"], nc["base"], nc["depth"], L, W_)
        ax.add_patch(_Poly2(tri, closed=True, fc="white",
                            ec=("#e08e0b" if i in bad_n else "#c0392b"),
                            lw=1.7, ls=((0, (4, 2)) if i in bad_n else "solid")))
        xs += list(tri[:, 0]); ys += list(tri[:, 1])
    for k, hl in enumerate(holes_mm, 1):
        if hl["diameter"] <= 0:
            continue                                  # disabled hole row
        r = hl["diameter"]/2.0
        ax.add_patch(_Circ2((hl["x"], hl["y"]), r, fc="white",
                            ec=("#e08e0b" if k in bad_h else "#c0392b"),
                            lw=1.7, ls=((0, (4, 2)) if k in bad_h else "solid")))
        xs += [hl["x"]-r, hl["x"]+r]; ys += [hl["y"]-r, hl["y"]+r]

    ax.plot(0, 0, "ko", ms=4)
    ax.set_aspect("equal")
    mx, my = 0.06*L, 0.06*W_
    ax.set_xlim(min(xs)-mx, max(xs)+mx); ax.set_ylim(min(ys)-my-0.06*W_, max(ys)+my)
    ax.set_xlabel("x [mm]"); ax.set_ylabel("y [mm]")
    n_holes = sum(1 for hl in holes_mm if hl["diameter"] > 0)
    n_bad = len(bad_n) + len(bad_h)
    extra = f"  —  {n_bad} off-bounds" if n_bad else ""
    ax.set_title(f"Geometry preview — {L_mm:g} × {W_mm:g} × {H_mm:g} mm, "
                 f"{len(notches_mm)} notch(es), {n_holes} hole(s){extra}")
    display(fig)

    probs = nprobs + hprobs
    if probs:
        print("⚠  these cuts won't mesh yet (shown orange-dashed):")
        for _, m in probs:
            print("   " + m)

In [ ]:
# Close widgets created by a previous run of THIS cell, so their comms are torn down and a
# single click can't be delivered to stale buttons (avoids a click running 2-3x after several
# "Run All"s without a kernel restart).
for _n in ('panel', 'tabs', 'out', 'preview_out', 'run_btn', 'status', 'w_L', 'w_W', 'w_H',
           'w_mat', 'w_E', 'w_rho', 'w_nu', 'w_fmax', 'w_nm', 'w_epw', 'w_incut', 'w_cmp',
           'w_notch', 'w_holes',
           'w_h1x', 'w_h1y', 'w_h1d', 'w_h2x', 'w_h2y', 'w_h2d'):
    _old = globals().get(_n)
    if _old is not None:
        try:
            _old.close()
        except Exception:
            pass

_S  = {'description_width': '92px'}
_LW = W.Layout(width='220px')
_SW = {'description_width': '165px'}             # wider labels for the material & solver tabs
_LWW = W.Layout(width='305px')

def _cap(text):
    return W.HTML(f"<span style='color:#666;font-size:90%'>{text}</span>")

# --- plate geometry --- (the preview redraws when you COMMIT a field: press Enter or click
# out of it. continuous_update is intentionally OFF: with it on, the field goes blank mid-typing
# because a half-typed number isn't valid.)
w_L = W.BoundedFloatText(value=300.0, min=1, max=5000, step=10, description='Length [mm]', style=_S, layout=_LW)
w_W = W.BoundedFloatText(value=100.0, min=1, max=5000, step=10, description='Width [mm]',  style=_S, layout=_LW)
w_H = W.BoundedFloatText(value=10.0,  min=0.5, max=1000, step=1, description='Thick. [mm]', style=_S, layout=_LW)

# --- material (E/rho/nu editable only for 'custom') ---
w_mat = W.Dropdown(options=['steel', 'bronze', 'aluminum', 'custom'], value='steel',
                   description='Material', style=_SW, layout=_LWW)
w_E   = W.BoundedFloatText(value=200.0, min=1, max=1000,  step=1,  description="Young's modulus [GPa]", style=_SW, layout=_LWW)
w_rho = W.BoundedFloatText(value=7800., min=100, max=25000, step=50, description='Density [kg/m³]',      style=_SW, layout=_LWW)
w_nu  = W.BoundedFloatText(value=0.30,  min=0.0, max=0.49, step=0.01, description="Poisson's ratio",     style=_SW, layout=_LWW)

# --- solver ---
w_fmax = W.BoundedFloatText(value=2000.0, min=100, max=50000, step=100, description='Max frequency [Hz]', style=_SW, layout=_LWW)
w_nm   = W.BoundedIntText(value=6, min=1, max=12, description='Number of modes', style=_SW, layout=_LWW)
w_epw  = W.BoundedIntText(value=8, min=4, max=16, description='Elements per wavelength', style=_SW, layout=_LWW)
w_cmp  = W.Checkbox(value=True, description='Compare with vs without cuts', indent=False)

# --- incuts ---
w_incut = W.Checkbox(value=True, description='Enable incuts', indent=False)
w_notch = W.Textarea(
    value="bottom, 75, 25, 15\nbottom, 225, 25, 15\ntop, 75, 25, 15\ntop, 225, 25, 15\n"
          "left, 25, 20, 15\nleft, 75, 20, 15\nright, 25, 20, 15\nright, 75, 20, 15",
    layout=W.Layout(width='330px', height='150px'))

# --- holes (two fixed rows; diameter 0 disables a hole) ---
_Sh  = {'description_width': '14px'}
_LWh = W.Layout(width='118px')
w_holes = W.Checkbox(value=False, description='Enable holes', indent=False)
w_h1x = W.BoundedFloatText(value=30.0,  min=0, max=5000, step=5, description='x', style=_Sh, layout=_LWh)
w_h1y = W.BoundedFloatText(value=20.0,  min=0, max=5000, step=5, description='y', style=_Sh, layout=_LWh)
w_h1d = W.BoundedFloatText(value=12.0,  min=0, max=5000, step=1, description='ø', style=_Sh, layout=_LWh)
w_h2x = W.BoundedFloatText(value=30.0,  min=0, max=5000, step=5, description='x', style=_Sh, layout=_LWh)
w_h2y = W.BoundedFloatText(value=80.0,  min=0, max=5000, step=5, description='y', style=_Sh, layout=_LWh)
w_h2d = W.BoundedFloatText(value=12.0,  min=0, max=5000, step=1, description='ø', style=_Sh, layout=_LWh)

run_btn = W.Button(description='▶ Run', button_style='success',
                   layout=W.Layout(width='150px', height='40px'))
status      = W.HTML("")
out         = W.Output()
preview_out = W.Output()

def _holes_mm():
    '''Gather the two hole rows (mm). Rows with diameter <= 0 are ignored downstream.'''
    return [dict(x=w_h1x.value, y=w_h1y.value, diameter=w_h1d.value),
            dict(x=w_h2x.value, y=w_h2y.value, diameter=w_h2d.value)]

# enable E/rho/nu only for custom; otherwise show the preset values (read-only)
def _sync_material(*_):
    custom = (w_mat.value == 'custom')
    for wdg in (w_E, w_rho, w_nu):
        wdg.disabled = not custom
    if not custom:
        p = MATERIALS[w_mat.value]
        w_E.value, w_rho.value, w_nu.value = p['E']/1e9, p['rho'], p['nu']
w_mat.observe(_sync_material, names='value')
_sync_material()

# live geometry preview: redraw whenever a geometry input changes (no meshing/solving).
# Only a MALFORMED incut line blocks drawing; out-of-bounds notches/holes are drawn + flagged.
def _refresh_preview(*_):
    with preview_out:
        preview_out.clear_output(wait=True)
        try:
            notches = parse_notches(w_notch.value) if w_incut.value else []
        except ValueError as e:
            print(f"⚠  can't read the incut list — {e}")
            return
        holes = _holes_mm() if w_holes.value else []
        draw_geometry_preview(w_L.value, w_W.value, w_H.value, notches, holes)

for _w in (w_L, w_W, w_H, w_incut, w_notch, w_holes,
           w_h1x, w_h1y, w_h1d, w_h2x, w_h2y, w_h2d):
    _w.observe(_refresh_preview, names='value')

_running = {'busy': False}

def _on_run(_):
    if _running['busy']:                      # ignore duplicate/re-entrant clicks
        return
    _running['busy'] = True
    run_btn.disabled = True; run_btn.description = '⏳ Running…'; run_btn.button_style = 'warning'
    status.value = "<span style='color:#b58900'>Running… meshing + solving</span>"
    try:
        with out:
            out.clear_output(wait=True)
            try:
                notches = parse_notches(w_notch.value) if w_incut.value else []
                run_analysis(w_L.value, w_W.value, w_H.value, w_mat.value,
                             w_E.value*1e9, w_rho.value, w_nu.value,   # E entered in GPa -> Pa
                             w_fmax.value, w_nm.value, w_epw.value,
                             w_incut.value, notches,
                             enable_holes=w_holes.value, holes_mm=_holes_mm(),
                             do_compare=w_cmp.value)
                status.value = "<span style='color:#2aa198'>✓ Done</span>"
            except (ValueError, RuntimeError) as e:           # expected / friendly errors
                print(f"⚠  {e}")
                status.value = "<span style='color:#dc322f'>⚠ Input/geometry problem — see message</span>"
            except Exception:                                 # unexpected -> full traceback
                traceback.print_exc()
                status.value = "<span style='color:#dc322f'>✗ Unexpected error — see traceback</span>"
    finally:
        run_btn.disabled = False; run_btn.description = '▶ Run'; run_btn.button_style = 'success'
        _running['busy'] = False

run_btn.on_click(_on_run)

# --- tabbed input panel -------------------------------------------------------------------
_hole_lbl = W.Layout(width='52px')

_tab_geom = W.VBox([
    _cap('Overall size of the rectangular plate (mm). Length runs along x, width along y, '
         'thickness along z.'),
    w_L, w_W, w_H,
])
_tab_mat = W.VBox([
    _cap('The metal the plate is made of. Pick a preset, or choose “custom” to type your own '
         'values — the three fields below unlock only then.'),
    w_mat, w_E, w_rho, w_nu,
])
_tab_solver = W.VBox([
    _cap('Highest frequency the mesh is sized to resolve, how many elastic modes to report, and '
         'how finely each bending wave is meshed (higher = more accurate, slower).'),
    w_fmax, w_nm, w_epw,
    w_cmp,
    _cap('Tick to also solve the plain (uncut) plate and tabulate how much the cuts shift each '
         'frequency. Costs one extra solve.'),
])
_tab_incuts = W.VBox([
    _cap('Triangular V-notches cut into the edges, one per line: edge, pos, base, depth (mm). '
         'edge ∈ bottom / top / left / right; pos is the distance along that edge.'),
    w_incut,
    w_notch,
])
_tab_holes = W.VBox([
    _cap('Round through-holes. Give each centre (x, y) measured from the (0,0) corner and a '
         'diameter ø (mm). Set ø = 0 to switch a hole off.'),
    w_holes,
    W.HBox([W.HTML("<b>Hole 1</b>", layout=_hole_lbl), w_h1x, w_h1y, w_h1d]),
    W.HBox([W.HTML("<b>Hole 2</b>", layout=_hole_lbl), w_h2x, w_h2y, w_h2d]),
])

tabs = W.Tab(children=[_tab_geom, _tab_mat, _tab_solver, _tab_incuts, _tab_holes])
for _i, _t in enumerate(['Plate geometry', 'Material properties', 'Solver settings',
                         'Triangular incuts', 'Holes']):
    tabs.set_title(_i, _t)

panel = W.VBox([
    W.HTML("<h3 style='margin:2px'>Bell-plate modal analysis</h3>"
           "<span style='color:#666'>Set parameters across the tabs — the geometry preview "
           "updates as you edit (press Enter or click out of a field) — then press ▶ Run.</span>"),
    tabs,
    _cap('<b style="font-size:108%">Geometry preview</b>  '
         '(updates when you press Enter / click out of a field, before any meshing)'),
    preview_out,
    W.HBox([run_btn, status]),
])
_refresh_preview()       # draw the initial preview
display(panel, out)